In [1]:
import pandas as pd
import numpy as np
import math
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

In [21]:
# # Инициализация веб-драйвера
# driver = webdriver.Chrome()

# # URL для отправки запроса
# url = "https://points.worldsdc.com/lookup2020"

# # Создаем список для хранения информации о каждом танцоре
# soup_list = []

# batch_size = 1000  # Размер пакета для обработки
# failed_requests = 0  # Счетчик неудачных запросов
# last_successful_request = 0  # Номер последнего успешного запроса

# # Переменная для хранения номера последнего удачного запроса перед 5 неудачными
# last_successful_before_failures = 0

# while failed_requests < 5:
#     try:
#         batch = list(map(str, range(last_successful_request + 1, last_successful_request + 1 + batch_size)))
        
#         for dancer_id in batch:
#             driver.get(url)
#             input_field = WebDriverWait(driver, 10).until(
#                 EC.presence_of_element_located((By.ID, "q"))
#             )
#             input_field.clear()
#             input_field.send_keys(dancer_id)
#             form = WebDriverWait(driver, 10).until(
#                 EC.presence_of_element_located((By.ID, "lookup_form"))
#             )
#             form.submit()
#             element = WebDriverWait(driver, 10).until(
#                 EC.text_to_be_present_in_element((By.ID, 'lookup_results'), dancer_id)
#             )
#             html_content = driver.page_source
#             soup = BeautifulSoup(html_content, "html.parser")
            
#             # Добавляем soup в список для каждого танцора независимо от результата
#             soup_list.append(soup)
            
#             last_successful_request = dancer_id
#             last_successful_before_failures = dancer_id
                
#     except Exception as e:
#         print(f"Ошибка при обработке танцора {dancer_id}: {e}")
#         soup_list.append(None)  # Сохраняем None в случае ошибки
#         failed_requests += 1 
        
#     time.sleep(1)


# # Закрываем драйвер после завершения цикла
# if driver:
#     driver.quit()

Ошибка при обработке танцора 11: Message: 

Ошибка при обработке танцора 11: can only concatenate str (not "int") to str
Ошибка при обработке танцора 11: can only concatenate str (not "int") to str
Ошибка при обработке танцора 11: can only concatenate str (not "int") to str
Ошибка при обработке танцора 11: can only concatenate str (not "int") to str


In [32]:
# Инициализация веб-драйвера
driver = webdriver.Chrome()

# URL для отправки запроса
url = "https://points.worldsdc.com/lookup2020"

max_attempts = 5  # Максимальное количество неудачных попыток
attempts = 0  # Счетчик неудачных попыток
dancer_id = 22735  # Номер танцора для начала проверки

while attempts < max_attempts:
    try:
        driver.get(url)
        input_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "q"))
        )
        input_field.clear()
        input_field.send_keys(str(dancer_id))
        form = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "lookup_form"))
        )
        form.submit()
        WebDriverWait(driver, 10).until(
            EC.text_to_be_present_in_element((By.ID, 'lookup_results'), str(dancer_id))
        )
        print(f"Информация найдена для танцора с номером: {dancer_id}")
        attempts = 0  # Обнуляем счетчик при успешной попытке
    except Exception as e:
        print(f"Для танцора с номером {dancer_id} информация не найдена.")
        attempts += 1  # Увеличиваем счетчик неудачных попыток
    dancer_id += 1  # Переходим к следующему номеру танцора

    # Если достигнуто максимальное количество неудачных попыток подряд, прерываем цикл
    if attempts == max_attempts:
        break

Информация найдена для танцора с номером: 22735
Информация найдена для танцора с номером: 22736
Для танцора с номером 22737 информация не найдена.
Для танцора с номером 22738 информация не найдена.
Для танцора с номером 22739 информация не найдена.
Для танцора с номером 22740 информация не найдена.
Для танцора с номером 22741 информация не найдена.


In [30]:
# у нас есть файл, который содержит сведения о значениях, где нет информации о танцоре, и парсить там нечего
def read_from_file(filename):
    with open(filename, 'r') as file:
        data = file.read().splitlines()
    return data

# Читаем список из файла
retrieved_empty_ids = read_from_file('empty_dancer_ids.txt')

In [3]:
# Создаем список для хранения информации о каждом танцоре
soup_list = []

# Максимальное количество танцоров для обработки
max_dancer_id = dancer_id - 1

# пройдем циклом от 1 до последнего танцора, пропуская номера, под которыми танцоров в базе нет
for dancer_id in range(1, dancer_id):
    if dancer_id not in retrieved_empty_ids:
        # Переход на страницу
        driver.get(url)

        try:
            # Ожидание появления поля ввода для учетного номера танцора
            input_field = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "q"))
            )
            input_field.clear()  # Очистка поля ввода перед вводом нового номера
            input_field.send_keys(str(dancer_id))  # Ввод номера танцора

            # Находим форму с кнопкой "Найти танцора" и выполняем отправку
            form = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "lookup_form"))
            )
            form.submit()

            # Ждем появления элемента с нужным текстом внутри блока lookup_results
            element = WebDriverWait(driver, 10).until(
                EC.text_to_be_present_in_element((By.ID, 'lookup_results'), str(dancer_id))
            )

            # Получаем HTML-содержимое страницы после выполнения запроса
            html_content = driver.page_source

            soup = BeautifulSoup(html_content, "html.parser")
            
            # Добавляем soup в список для каждого танцора
            soup_list.append(soup)
            
        except Exception as e:
            print(f"Ошибка при обработке танцора {dancer_id}: {e}")       

# Закрываем драйвер после завершения цикла
if driver:
    driver.quit()

Ошибка при обработке танцора 11: Message: 

Ошибка при обработке танцора 23: Message: 

Ошибка при обработке танцора 44: Message: 

Ошибка при обработке танцора 45: Message: 

Ошибка при обработке танцора 101: Message: 

Ошибка при обработке танцора 161: Message: 

Ошибка при обработке танцора 166: Message: 

Ошибка при обработке танцора 201: Message: 

Ошибка при обработке танцора 286: Message: 

Ошибка при обработке танцора 336: Message: 

Ошибка при обработке танцора 347: Message: 

Ошибка при обработке танцора 362: Message: 

Ошибка при обработке танцора 366: Message: 

Ошибка при обработке танцора 420: Message: 

Ошибка при обработке танцора 431: Message: 

Ошибка при обработке танцора 463: Message: 

Ошибка при обработке танцора 464: Message: 

Ошибка при обработке танцора 465: Message: 

Ошибка при обработке танцора 467: Message: 

Ошибка при обработке танцора 473: Message: 

Ошибка при обработке танцора 493: Message: 

Ошибка при обработке танцора 498: Message: 

Ошибка при обр

In [4]:
def extract_role_info(soup, dancer_id):
    roles = {
        'dancer_id': dancer_id,
        'dancer_name': None,
        'primary_role': None,
        'primary_role_down_class': None,
        'primary_role_up_class': None,
        'secondary_role': None,
        'secondary_role_down_class': None,
        'secondary_role_up_class': None,
    }

    role_info = soup.find_all('p', class_='lead')

    for info in role_info:
        strong_tags = info.find_all('strong')
        for strong_tag in strong_tags:
            role_name = strong_tag.text.strip().replace(':', '')
            label_tags = strong_tag.find_next_siblings('span', class_='label')
            if label_tags:
                label_texts = []
                for tag in label_tags:
                    label_text = re.sub(r'\s+', ' ', tag.text.strip())
                    label_texts.append(label_text)
                if "Primary Role Leader Dance Level" in role_name:
                    roles['primary_role_down_class'] = label_texts[0]
                    roles['primary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['primary_role'] = 'leader'
                elif "Primary Role Follower Dance Level" in role_name:
                    roles['primary_role_down_class'] = label_texts[0]
                    roles['primary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['primary_role'] = 'follower'
                elif "Secondary Role Leader Dance Level" in role_name:
                    roles['secondary_role_down_class'] = label_texts[0]
                    roles['secondary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['secondary_role'] = 'leader'
                elif "Secondary Role Follower Dance Level" in role_name:
                    roles['secondary_role_down_class'] = label_texts[0]
                    roles['secondary_role_up_class'] = label_texts[1] if len(label_texts) > 1 else np.nan
                    roles['secondary_role'] = 'follower'

    dancer_name = soup.find('h1').text.strip()
    roles['dancer_name'] = dancer_name

    return roles

# Создание списка информации о танцорах
all_dancers_info = []


import numpy as np

for dancer_id in range(1, max_dancer_id + 1):
    found = False
    for soup in soup_list:
        try:
            found_dancer_id = int(soup.find('h1').text.strip().split()[-1][1:-1])
        except ValueError:
            found_dancer_id = None  # Если не удалось преобразовать в число, присвоить None
        
        if found_dancer_id == dancer_id:
            dancer_role_info = extract_role_info(soup, dancer_id)
            all_dancers_info.append(dancer_role_info)
            found = True
            break
    
    if not found:
        roles = {'dancer_id': dancer_id}
        for key in ['dancer_name', 'primary_role', 'primary_role_down_class', 'primary_role_up_class',
                    'secondary_role', 'secondary_role_down_class', 'secondary_role_up_class']:
            roles[key] = np.nan
        all_dancers_info.append(roles)


# Создание DataFrame из списка информации о танцорах
dancer_role_info_df = pd.DataFrame(all_dancers_info)
# Упорядочивание столбцов
dancer_role_info = dancer_role_info_df[['dancer_id', 'dancer_name', 'primary_role', 'primary_role_down_class', 
                                        'primary_role_up_class','secondary_role', 'secondary_role_down_class', 
                                        'secondary_role_up_class']]

dancer_role_info['dancer_name'] = dancer_role_info['dancer_name'].str.replace('\t', '')


In [27]:
empty_names = dancer_role_info[dancer_role_info['dancer_name'].isna()]
empty_dancer_ids = empty_names['dancer_id'].tolist()

In [29]:
# Функция для сохранения списка в файл
def save_to_file(filename, data):
    with open(filename, 'w') as file:
        for item in data:
            file.write(str(item) + '\n')

save_to_file('empty_dancer_ids.txt', empty_dancer_ids)

In [5]:
def extract_dancer_info(soup):
    dancer_info = []
    
    dancer_name = soup.find('h1').text.strip()
    match = re.search(r'\((\d+)\)', dancer_name)  # Ищем цифры в кавычках
    if match:
        dancer_name = match.group(1)  # Оставляем только цифры в кавычках
    
    h2_elements = soup.find_all('h2')
    
    for h2 in h2_elements:
        h2_text = h2.text.strip().split('-')[-1].strip()  # Оставляем только последнюю часть в role_dancer
        
        h3_elements = h2.find_all_next('h3')  # Находим все элементы h3 после текущего h2
        
        if h3_elements:
            for h3 in h3_elements:
                h3_text = h3.text.strip().split()[0]  # Оставляем только первую часть в class_dancer
                points = h3.find('span', class_='label').text.strip().split()[0]
                dancer_info.append([dancer_name, h2_text, h3_text, points])
        else:
            dancer_info.append([dancer_name, h2_text, None, None])
    
    columns = ['dancer_id', 'role_dancer', 'class_dancer', 'points']
    dancer_df = pd.DataFrame(dancer_info, columns=columns)
    return dancer_df

# Пример использования для списка soup_list
dancers_points_info = pd.DataFrame()

for soup in soup_list:
    dancer_info_df = extract_dancer_info(soup)
    dancers_points_info = pd.concat([dancers_points_info, dancer_info_df], ignore_index=True)
    



In [29]:
dancers_points_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40802 entries, 0 to 40801
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   dancer_id     40802 non-null  object
 1   role_dancer   40802 non-null  object
 2   class_dancer  40802 non-null  object
 3   points        40802 non-null  object
dtypes: object(4)
memory usage: 1.2+ MB


In [6]:
def extract_dancer_results(soup):
    dancer_results = []

    h1 = soup.find('h1')
    dancer_id_match = re.search(r'\((\d+)\)', h1.text.strip())  # Находим цифры в кавычках
    if dancer_id_match:
        dancer_id = dancer_id_match.group(1)
    
    h2_elements = soup.find_all('h2')
    
    for h2 in h2_elements:
        role_dancer = h2.text.strip().split('-')[-1].strip()  # Оставляем только последнее слово в role_dancer
        
        h3_elements = h2.find_all_next('h3')  # Находим все элементы h3 после текущего h2
        
        if h3_elements:
            for h3 in h3_elements:
                class_dancer = h3.text.strip().split()[0]  # Оставляем только первую часть в class_dancer
                table = h3.find_next('table', class_='table-striped')
                if table:
                    rows = table.find_all('tr')
                    for row in rows[1:]:  # Начинаем с 1, чтобы пропустить заголовок таблицы
                        row_data = [dancer_id, role_dancer, class_dancer] + [cell.text.strip() for cell in row.find_all('td')]
                        dancer_results.append(row_data)
    
    columns = ['dancer_id', 'role_dancer', 'class_dancer', 'Role', 'Event', 'Location', 'Date', 'Result', 'Points']
    dancer_results_df = pd.DataFrame(dancer_results, columns=columns)
    return dancer_results_df

# Пример использования для списка soup_list
dancers_results_info = pd.DataFrame()

for soup in soup_list:
    dancer_results_df = extract_dancer_results(soup)
    dancers_results_info = pd.concat([dancers_results_info, dancer_results_df], ignore_index=True)

In [31]:
dancers_results_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155816 entries, 0 to 155815
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   dancer_id     155816 non-null  object
 1   role_dancer   155816 non-null  object
 2   class_dancer  155816 non-null  object
 3   Role          155816 non-null  object
 4   Event         155816 non-null  object
 5   Location      155816 non-null  object
 6   Date          155816 non-null  object
 7   Result        155816 non-null  object
 8   Points        155816 non-null  object
dtypes: object(9)
memory usage: 10.7+ MB


In [23]:
dancer_role_info.to_csv('dancer_role_info.csv', index=False)
dancers_points_info.to_csv('dancers_points_info.csv', index=False)
dancers_results_info.to_csv('dancers_results_info.csv', index=False)

In [ ]:
dancer_role_info[]